In [1]:
import os
kaggle = False
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'kaggle' in dirname:
            kaggle = True 

In [ ]:
SEED = 42
import tensorflow as tf
tf.random.set_seed(SEED)

import numpy as np
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)

from sklearn.model_selection import train_test_split
import array
import json
import bz2
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertConfig, BertForSequenceClassification, get_linear_schedule_with_warmup, BertModel
from torch.utils.data import DataLoader, IterableDataset
from accelerate import Accelerator
import time
import re
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import sys
from tqdm.auto import tqdm
from torchsummary import summary
import math
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, classification_report
from safetensors import safe_open
from sklearn.utils.class_weight import compute_class_weight

from classes import Cust_FineTune_BertClass
import seaborn as sns

In [ ]:
MAX_LENGTH = 202
UNIQUE_CHAR = ['!',	'#', '$', '&', "'", '(', ')', '*', '+', ',', '/', ':', ';', '=', '?', '@', '[', ']', '%',
               '-', '_', '~', '.',
               'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 
               'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',                
               '0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
SPECIAL_TOKENS = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]', '[SEC]', '[ISEC]']

UNIQUE_CHAR_LEN = len(UNIQUE_CHAR)
print(UNIQUE_CHAR_LEN)


vocab =  UNIQUE_CHAR + SPECIAL_TOKENS
token_to_id  = {char: idx for idx, char in enumerate(vocab)}
id_to_token = {idx: char for char, idx in token_to_id.items()}
vocab_size = len(vocab)

In [ ]:

if kaggle:
    file_path = '/kaggle/input/ft-dataset/ft_dataset.json'
else:
    file_path = 'datasets/ft_dataset.json'
    # file_path = 'datasets/phishing_and_benign_websites.csv'

def load_classification_data_bz2(file_path):
    # urls, labels = [], []
    # with bz2.open(file_path, 'rt') as f:
    #     for line in f:
    #         data = json.loads(line)
    #         if len(data["url"]) < 200:            
    #             urls.append(data["url"])
    #             labels.append(data["phish"])
    df = pd.read_json(file_path, compression='bz2', lines=True)
    # return urls, labels
    return df

def load_classification_data(file_path):
    df = pd.read_json(file_path, lines=True)
    return df

def load_classification_data_csv(file_path):
    df = pd.read_csv(file_path)
    return df

# ft_urls1, labels = load_classification_data(file_path)
# ft_urls2 = [item.lower() for item in ft_urls1]

# if kaggle:
#     data = load_classification_data(file_path)
# else:
#     data = load_classification_data_bz2(file_path)

if file_path.endswith('.bz2'):
    data = load_classification_data(file_path)
    X = data['url']
    y = data['phish']
    test_size1=0.2
    test_size2=0.5
    train_size1= 1 - test_size1
    train_size2= 1 - test_size2  
        
elif file_path.endswith('.json'):
    data = load_classification_data(file_path)
    X = data['url']
    y = data['phish']
    test_size1=0.2
    test_size2=0.5
    train_size1= 1 - test_size1
    train_size2= 1 - test_size2   
        
elif file_path.endswith('.csv'):
    data = load_classification_data_csv(file_path)
    data['Label'] = data['Label'].replace({'Phishing': 1, 'Benign': 0})
    X = data['URLs']
    y = data['Label']
    train_size1=4/7
    train_size2=2/3
    test_size1= 1 - train_size1
    test_size2= 1 - train_size2     

    
# data['url'] = data['url'].str.lower()
# X = data['url']
# y = data['phish']

# df = pd.read_csv('datasets/phishing_and_benign_websites.csv')
# df['Label'] = df['Label'].replace({'Phishing': 1, 'Benign': 0})

# X = df['URLs']
# y = df['Label']

# print(data['Label'].value_counts())

X_train, X_temp, y_train, y_temp = train_test_split(
                                                    X, 
                                                    y, 
                                                    # train_size=4/7,
                                                    # test_size=0.2,
                                                    # train_size=train_size1,
                                                    test_size=test_size1,                                                    
                                                    random_state=42,
                                                    shuffle=True, 
                                                    stratify=y,
                                                    )

X_test, X_val, y_test, y_val = train_test_split(
                                                X_temp, 
                                                y_temp, 
                                                # train_size=2/3,
                                                # test_size=0.5,
                                                # train_size=train_size2,
                                                test_size=test_size2,                                                
                                                random_state=42,
                                                shuffle=True, 
                                                stratify=y_temp,
                                                )

# del data
# del pretrain_urls1
# del pretrain_urls2

class_wts = compute_class_weight('balanced', classes=np.unique(y), y=y)
print(class_wts)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(X_val.shape)
print(y_val.shape)

In [5]:
def fine_tune_preprocess(urls, phish):
    for i in range(len(urls)):
        url = urls.iloc[i]
        labels = phish.iloc[i]
        # if labels == 0:
        #     continue
            
        # print(url)
        x = url.partition('//')[2]

        # print(x)
        
        is_secure = url.partition('//')[0]

        # print(is_secure)
        
        if is_secure == 'https:':
            is_secure = '[SEC]'
        else:
            is_secure = '[ISEC]'
        tokens = ['[CLS]'] + [is_secure] + list(x)
        
        # print(tokens)
        
        if len(tokens) > MAX_LENGTH:
            tokens = tokens[:MAX_LENGTH-1] + [tokens[-1]]
            
        # Convert to token IDs
        input_ids = [token_to_id.get(token, token_to_id['[UNK]']) for token in tokens]

        # print(input_ids)
        
        # Pad to MAX_LENGTH
        if len(input_ids) < MAX_LENGTH:
            padding = [token_to_id['[PAD]']] * (MAX_LENGTH - len(input_ids))
            input_ids += padding
        
        # Create attention mask
        attention_mask = [1] * len(tokens) + [0] * (MAX_LENGTH - len(tokens))

        # print(attention_mask)
        # labels = phish[(urls==url).argmax()]
        # print(type(urls))

        # print(labels)
        # break
        yield {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels, dtype=torch.long),
            # "token_type_ids": torch.tensor(torch.zeros_like(input_ids))
        }


In [6]:
class finetune_dataset(IterableDataset):
    def __init__(self, urls, phish): 
        super().__init__()
        self.urls = urls
        self.phish = phish        

    def __iter__(self):
        return fine_tune_preprocess(self.urls, self.phish) 

train_dataset = finetune_dataset(X_train, y_train)
val_dataset = finetune_dataset(X_val, y_val)
test_dataset = finetune_dataset(X_test, y_test)

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    pin_memory = True, 

)

val_loader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    pin_memory = True,
    
)

test_loader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    pin_memory = True,

)

In [ ]:
# num_epochs = 10
num_epochs = 20

device = torch.device('cuda')
model = Cust_FineTune_BertClass(pretrained_path="run4_pretrain.pt/model.safetensors", num_labels=2)
# model = Cust_FineTune_BertClass(pretrained_path=None, num_labels=2)

model = model.to(device)

optimizer = torch.optim.AdamW([
                            {'params': model.bert.parameters(), 'lr': 5e-6},  #2e-5 1e-5  5e-1
                            {'params': model.pre_classifier.parameters(), 'lr': 5e-6}, #2e-4  1e-4
                            {'params': model.classifier.parameters(), 'lr': 5e-6}], 
                            weight_decay=0.01, #0.01
                            betas=(0.9, 0.999), # none, .5  
                            )

weights= torch.tensor(class_wts,dtype=torch.float)
weights = weights.to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights)

total_steps = math.ceil(len(X_train) / batch_size) * num_epochs
warmup_steps = int(0.05 * total_steps)  
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
            
class EarlyStoppingf1:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_f1 = None
        self.early_stop = False
        
    def __call__(self, f1):
        if self.best_f1 is None:
            self.best_f1 = f1
        elif f1 < self.best_f1 - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_f1 = f1
            self.counter = 0            
            
# early_stopping = EarlyStopping(patience=5, min_delta=0.001)      
early_stopping = EarlyStoppingf1(patience=3, min_delta=0.001)                  

In [ ]:
epochs_val_accuracy = []
epochs_accuracy = []

epochs_losses = []
epochs_val_losses = []

epochs_val_precision = [] 
epochs_val_recall = []
epochs_val_f1 = []
epochs_val_fpr = []

lr = []

best_val_loss = 0.0
best_f1 = 0.0

for epoch in range(num_epochs):
    # Training
    model.train()
    
    num_batch = 0
    total_loss = 0
    train_preds = []
    train_labels = []

    progress_bar = tqdm(train_loader, total=math.ceil(len(X_train) / batch_size), desc=f"Train Epoch {epoch+1}", unit="batch")
    for batch in progress_bar:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # for item in batch.items():
        #     print(item)
        
        
        # Zero gradients
        optimizer.zero_grad()        
        
        # inputs = {
        #     "input_ids": batch["input_ids"],
        #     "attention_mask": batch["attention_mask"],
        #     "labels": batch["labels"],
        # }
        
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )
        labels = batch['labels'].to(device)

        # outputs = model(**inputs)       
        
        # logits = outputs
        loss = criterion(logits, batch["labels"])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        scheduler.step()
        
        if num_batch % 100 == 0:
            lr.append(optimizer.param_groups[0]['lr'])
        
        num_batch += 1
        
        total_loss += loss.item()
        avg_train_loss = total_loss / num_batch
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(labels.cpu().numpy())        
    
        # progress_bar.set_postfix({'loss': loss.item()})
        progress_bar.set_postfix({
            "avg_loss": f"{(total_loss/num_batch):.4f}",
            "batches": f"{num_batch}"
        })        
    progress_bar.close()    
    
    epochs_losses.append(avg_train_loss)
    
    train_accuracy = accuracy_score(train_labels, train_preds)
    epochs_accuracy.append(train_accuracy)
    
    # Validation
    model.eval()
    val_preds = []
    val_labels = []
    val_loss = 0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, total=math.ceil(len(X_val) / batch_size), desc=f" Validation Epoch {epoch+1}", unit="batch")
        val_batch = 0
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            
            # logits = outputs.logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            
            val_preds.extend(preds)
            val_labels.extend(labels.cpu().numpy())
            val_batch += 1
    
    progress_bar.close()   
     
    avg_val_loss = val_loss / val_batch
    epochs_val_losses.append(val_loss / val_batch)
        
    val_accuracy = accuracy_score(val_labels, val_preds)
    epochs_val_accuracy.append(val_accuracy)
    
    precision = precision_score(val_labels, val_preds)
    epochs_val_precision.append(precision)
    
    recall = recall_score(val_labels, val_preds)
    epochs_val_recall.append(recall)
    
    f1 = f1_score(val_labels, val_preds)
    epochs_val_f1.append(f1)
    
    tn, fp, fn, tp = confusion_matrix(val_labels, val_preds).ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    epochs_val_fpr.append(fpr)
    
    # if avg_val_loss < best_val_loss:
    #     best_val_loss = avg_val_loss
    #     torch.save(model.state_dict(), "finetune/best_model.pt")

    if best_f1 < f1:
        best_f1 = f1
        torch.save(model.state_dict(), "finetune/best_model.pt")
        print("Best model saved")
        
    print(f"Epoch {epoch+1} | Train Accuracy: {train_accuracy:.4f} | Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Epoch {epoch+1} | Accuracy: {val_accuracy:.4f}, Val Loss: {avg_val_loss:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")
        
    early_stopping(f1) 
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break
    
    # print(f"Epoch {epoch+1} | Train Accuracy: {train_accuracy:.4f} | Train Loss: {avg_train_loss:.4f}")
    # print(f"Validation Epoch {epoch+1} | Accuracy: {val_accuracy:.4f}, Val Loss: {avg_val_loss:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

# if kaggle:
#     torch.save(model.state_dict(), "/kaggle/working/finetune.pt")
# else:    
#     torch.save(model.state_dict(), "finetune/finetune.pt")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Cust_FineTune_BertClass(pretrained_path=None, num_labels=2)
model.load_state_dict(torch.load('finetune/past stats/finetune.pt', weights_only=True))
# model.load_state_dict(torch.load('finetune/fix ft/best_model.pt', weights_only=True))
# model.load_state_dict(torch.load('finetune//best_model.pt', weights_only=True))
# model.load_state_dict(torch.load('finetune/pbwd/best_model.pt', weights_only=True))
model.to(device)
# model.eval()
weights= torch.tensor(class_wts,dtype=torch.float)
weights = weights.to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights)

In [ ]:
model.eval()
test_preds = []
test_labels = []
test_probs = []
test_loss = 0

with torch.no_grad():
    progress_bar = tqdm(test_loader, total=math.ceil(len(X_test) / batch_size), desc=f"Test", unit="batch")
    test_batch = 0
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # if test_batch == 0:
        #     print(batch['input_ids'].shape, batch['attention_mask'].shape)
        #     break
        
        logits = model(input_ids, attention_mask=attention_mask)
        loss = criterion(logits, labels)
        test_loss += loss.item()
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        probabilities = torch.softmax(logits, dim=1)
        positive_probs = probabilities[:, 1].cpu().numpy()
        
        test_probs.extend(positive_probs)
        test_preds.extend(preds)
        test_labels.extend(labels.cpu().numpy())
        test_batch += 1

progress_bar.close()    
avg_test_loss = test_loss / test_batch if test_batch else 0.0
    
test_accuracy = accuracy_score(test_labels, test_preds)
test_precision = precision_score(test_labels, test_preds)
test_recall = recall_score(test_labels, test_preds)
test_f1 = f1_score(test_labels, test_preds)

test_tn, test_fp, test_fn, test_tp = confusion_matrix(test_labels, test_preds).ravel()
test_fpr = test_fp / (test_fp + test_tn) if (test_fp + test_tn) > 0 else 0

print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1 Score: {test_f1:.4f}")
print(f"FPR: {test_fpr:.4f}")


In [ ]:
print(classification_report(test_labels, test_preds, digits=4))

test_fnr = test_fn / (test_fn + test_tp) if (test_fn + test_tp) > 0 else 0
print(f"FNR: {test_fnr:.4f}")

confu = confusion_matrix(test_labels, test_preds)

group_names = ['True Negative','False Positive','False Negative','True Positive']
categories = ['Benign', 'Phishing']

group_counts = ["{0:0.0f}".format(value) for value in
                confu.flatten()]
group_percentages = ["{0:.2%}".format(value) for value in
                     confu.flatten()/np.sum(confu)]
labels = [f"{v1}\n{v2} ({v3})" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
labels = np.asarray(labels).reshape(2,2)


ax = sns.heatmap(
                    confu, 
                    annot=labels, 
                    fmt='', 
                    cmap='Blues', 
                    xticklabels=categories, 
                    yticklabels=categories,
                    )

ax.set(xlabel='Predicted Labels', ylabel='True Labels')
plt.savefig('finetune/fix ft/confu_matrix.png')
plt.savefig('finetune/pbwd/confu_matrix.png')
plt.show()

In [ ]:
actual_fpr = test_fp / (test_fp + test_tn)
actual_fpr_percentage = (test_fp / np.sum(confu)) * 100

print(f"Actual FPR: {actual_fpr:.6f}")
print(f"FPR as percentage: {actual_fpr_percentage:.4f}%")
print(f"Confusion matrix FPR percentage: {(test_fp/np.sum(confu))*100:.4f}%")

In [14]:
# fpr, tpr, _ = roc_curve(test_labels, test_probs)
# roc_auc = auc(fpr, tpr)

# plt.figure()
# plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# plt.title('Receiver Operating Characteristic')
# plt.legend(loc="lower right")
# plt.show()

# target_fpr = 0.0001  # 0.01%
# idx = np.argmin(np.abs(fpr - target_fpr))
# tpr_at_target_fpr = tpr[idx]
# actual_fpr = fpr[idx]

# print(f"TPR at FPR≈{actual_fpr*100:.4f}%: {tpr_at_target_fpr:.4f}")

In [15]:
i = 0
if kaggle:
    # try:
    #     with open("/kaggle/working/pretrain.pt/stats.txt", "x") as f:
    #         while i < len(epoch_losses):
    #             f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
    #             f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
    #             i = i + 1
    # except FileExistsError:
    #     with open("/kaggle/working/pretrain.pt/stats.txt", "w") as f:
    #         while i < len(epoch_losses):
    #             f.write(f"Epoch {i+1} | Train Loss: {epoch_losses[i]:.4f} | Train MLM Acc: {epoch_accuracies[i]:.4f} | Train NSP Acc: {epoch_nsp_accuracies[i]:.4f}\n")
    #             f.write(f"Epoch {i+1} | Val Loss: {epoch_val_losses[i]:.4f} | Val MLM Acc: {epoch_val_accuracies[i]:.4f} | Val NSP Acc: {epoch_val_nsp_accuracies[i]:.4f}\n")
    #             i = i + 1           
    pass     
else:    
    try:
        with open("finetune/stats.txt", "x") as f:
            while i < len(epochs_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epochs_losses[i]:.4f} | Train Acc: {epochs_accuracy[i]:.4f}\n")
                f.write(f"Validation Epoch {i+1} | Val Loss: {epochs_val_losses[i]:.4f} | Val Acc: {epochs_val_accuracy[i]:.4f} | Precision: {epochs_val_precision[i]:.4f} | Recall: {epochs_val_recall[i]:.4f} | F1: {epochs_val_f1[i]:.4f}\n")
                i = i + 1          
            f.write(f"Test | Loss: {avg_test_loss:.4f} | Acc: {test_accuracy:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f}\n")
    except FileExistsError:
        with open("finetune/stats.txt", "w") as f:
            while i < len(epochs_losses):
                f.write(f"Epoch {i+1} | Train Loss: {epochs_losses[i]:.4f} | Train Acc: {epochs_accuracy[i]:.4f}\n")
                f.write(f"Validation Epoch {i+1} | Val Loss: {epochs_val_losses[i]:.4f} | Val Acc: {epochs_val_accuracy[i]:.4f} | Precision: {epochs_val_precision[i]:.4f} | Recall: {epochs_val_recall[i]:.4f} | F1: {epochs_val_f1[i]:.4f}\n")
                i = i + 1     
            f.write(f"Test | Loss: {avg_test_loss:.4f} | Acc: {test_accuracy:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f}\n")

In [ ]:
plt.figure(figsize=(15, 10))

# Loss plot
plt.subplot(2, 2, 1)
plt.plot(range(1, len(epochs_losses)+1), epochs_losses, 'bo-', label='Train Loss')
plt.plot(range(1, len(epochs_val_losses)+1), epochs_val_losses, 'ro-', label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xlim(0, len(epochs_losses)+1)
plt.legend()
plt.grid(True)

# Accuracy plot
plt.subplot(2, 2, 2)
plt.plot(range(1, len(epochs_val_accuracy)+1), epochs_val_accuracy, 'ro-', label='Validation Accuracy')
plt.plot(range(1, len(epochs_accuracy)+1), epochs_accuracy, 'bo-', label='Train Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
# plt.ylim(0, 1)
plt.xlim(0, len(epochs_val_accuracy)+1)
plt.legend(loc="lower right")
plt.grid(True)

# Accuracy plot
plt.subplot(2, 2, 3)
plt.plot(range(1, len(epochs_val_precision)+1), epochs_val_precision, 'bo-', label='Validation Precision')
plt.plot(range(1, len(epochs_val_recall)+1), epochs_val_recall, 'ro-', label='Validation Recall')
plt.plot(range(1, len(epochs_val_f1)+1), epochs_val_f1, 'go-', label='Validation F1')
plt.title('Model Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
# plt.ylim(0, 1)
plt.xlim(0, len(epochs_val_accuracy)+1)
plt.legend(loc="lower right")
plt.grid(True)

# # Precision recall plot
# plt.subplot(2, 2, 4)
# plt.plot(epochs_val_precision, epochs_val_recall, 'bo-')
# plt.title('Precision recall curve')
# plt.xlabel('Precision')
# plt.ylabel('Recall')
# plt.ylim(0, 1)
# # plt.xlim(0, 1)
# plt.legend()
# plt.grid(True)

# Fpr
# plt.subplot(2, 2, 4)
# plt.plot(range(1, len(epochs_val_fpr)+1), epochs_val_fpr, 'bo-', label='Validation FPR')
# plt.title('FPR Curve')
# plt.xlabel('Epoch')
# plt.ylabel('FPR')
# # plt.ylim(0, 1)
# plt.xlim(0, len(epochs_val_fpr)+1)
# plt.legend()
# plt.grid(True)

# plt.subplot(2, 2, 4)
# plt.plot(range(1, len(lr)+1), lr, 'bo-', label='Validation FPR')
# plt.title('FPR Curve')
# plt.xlabel('Epoch')
# plt.ylabel('FPR')
# # plt.ylim(0, 1)
# # plt.xlim(0, len(epochs_val_fpr)+1)
# plt.legend()
# plt.grid(True)

fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)

plt.subplot(2, 2, 4)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")

plt.tight_layout()
plt.savefig('finetune/training_metrics.png')
plt.show()

In [ ]:
plt.figure()
plt.plot(range(1, len(epochs_val_fpr)+1), epochs_val_fpr, 'bo-', label='Validation FPR')
plt.title('FPR Curve')
plt.xlabel('Epoch')
plt.ylabel('FPR')
# plt.ylim(0, 1)
plt.xlim(0, len(epochs_val_fpr)+1)
plt.legend()
plt.grid(True)
plt.show()

In [51]:
# df = pd.read_csv('datasets/phishing_and_benign_websites.csv')

# df['Label'] = df['Label'].replace({'Phishing': 1, 'Benign': 0})

# X2 = df['URLs']
# y2 = df['Label']

# X_train, X_test2, y_train, y_test2 = train_test_split(
#                                                     X2, 
#                                                     y2, 
#                                                     test_size=0.4,
#                                                     random_state=42,
#                                                     shuffle=True, 
#                                                     stratify=y2,
# )

# # test_df = pd.DataFrame({'URLs': X_test2, 'Label': y_test2})
# # benign = test_df[test_df['Label'] == 0]
# # phishing = test_df[test_df['Label'] == 1]

# # n_phishing = len(phishing)
# # n_benign_required = 20 * n_phishing

# # if len(benign) > n_benign_required:
# #     benign = benign.sample(n_benign_required, random_state=42)
# # else:
# #     n_phishing = len(benign) // 20
# #     phishing = phishing.sample(n_phishing, random_state=42)

# # test_df_balanced = pd.concat([benign, phishing], ignore_index=True)
# # test_df_balanced = test_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# # X_test2_balanced = test_df_balanced['URLs']
# # y_test2_balanced = test_df_balanced['Label']

# # test2_dataset = finetune_dataset(X_test2_balanced, y_test2_balanced)
# test2_dataset = finetune_dataset(X_test2, y_test2)

# test2_loader = DataLoader(
#     test2_dataset,
#     batch_size = batch_size,
#     pin_memory = True, 
# )

# model.eval()
# test_preds2 = []
# test_labels2 = []
# test_loss2 = 0
    
# with torch.no_grad():
#     progress_bar = tqdm(test2_loader, total=math.ceil(len(X_test2) / batch_size), desc=f"Test2", unit="batch")
#     test_batch = 0
#     for batch in progress_bar:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['labels'].to(device)
        
#         # if test_batch == 0:
#         #     print(batch['input_ids'].shape, batch['attention_mask'].shape)
#         #     break
        
#         logits = model(input_ids, attention_mask=attention_mask)
#         loss = criterion(logits, labels)
#         test_loss2 += loss.item()
        
#         preds = torch.argmax(logits, dim=1).cpu().numpy()
        
#         test_preds2.extend(preds)
#         test_labels2.extend(labels.cpu().numpy())
#         test_batch += 1

# progress_bar.close()    
# avg_test_loss2 = test_loss2 / test_batch if test_batch else 0.0
    
# test_accuracy2 = accuracy_score(test_labels2, test_preds2)

# test_precision2 = precision_score(test_labels2, test_preds2)
# test_recall2 = recall_score(test_labels2, test_preds2)
# test_f12 = f1_score(test_labels2, test_preds2)

# print(f"Test Loss: {avg_test_loss2:.4f}")
# print(f"Test Accuracy: {test_accuracy2:.4f}")
# print(f"Precision: {test_precision2:.4f}")
# print(f"Recall: {test_recall2:.4f}")
# print(f"F1 Score: {test_f12:.4f}")